# Ceska sentiment analyza - CSFD recenze
Model trenovaný na recenzich z CSFD.cz

In [2]:
from google.colab import files
uploaded = files.upload()

Saving reviews_final.csv to reviews_final.csv


In [12]:
import pandas as pd

df = pd.read_csv('reviews_final.csv')
print(f'Celkem: {len(df)}')
print(df['sentiment'].value_counts())
df.head(3)

Celkem: 6589
sentiment
positive    2621
negative    2539
neutral     1429
Name: count, dtype: int64


,text,sentiment,film_id
0,bláznivá devadesátková krejzy komedie za soukr...,negative,9058
1,možná že tvůrci asi chtěli aby to byl jakýsi k...,negative,131733
2,tento film má něco do sebe přesto ale je hrozn...,negative,162926


In [19]:
from sklearn.utils import resample

df_pos = df[df['sentiment'] == 'positive']
df_neg = df[df['sentiment'] == 'negative']
df_neu = df[df['sentiment'] == 'neutral']

# Upsample neutral na stejný počet jako pos/neg
df_neu_up = resample(df_neu, replace=True, n_samples=len(df_pos), random_state=42)

df_balanced = pd.concat([df_pos, df_neg, df_neu_up]).sample(frac=1, random_state=42)
print(df_balanced['sentiment'].value_counts())

X = df_balanced['text']
y = df_balanced['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sentiment
neutral     2621
positive    2621
negative    2539
Name: count, dtype: int64


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ('clf', CalibratedClassifierCV(LinearSVC(max_iter=2000, C=5.0)))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'positive']))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8292

              precision    recall  f1-score   support

    negative       0.79      0.80      0.79       508
     neutral       0.89      0.84      0.87       525
    positive       0.81      0.85      0.83       524

    accuracy                           0.83      1557
   macro avg       0.83      0.83      0.83      1557
weighted avg       0.83      0.83      0.83      1557

Confusion matrix:
[[404  36  68]
 [ 47 443  35]
 [ 61  19 444]]


In [22]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-záčďéěíňóřšťúůýžÁČĎÉĚÍŇÓŘŠŤÚŮÝŽ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def predict_sentiment(text):
    cleaned = clean_text(text)
    pred = pipeline.predict([cleaned])[0]        # 'positive'/'negative'/'neutral'
    proba = pipeline.predict_proba([cleaned])[0]
    confidence = max(proba)
    score = int(1 + (confidence - 0.33) * 13.4)
    score = max(1, min(10, score))
    return pred.upper(), confidence, score

# Test
for t in testy2:
    label, conf, score = predict_sentiment(t)
    print(f'{label} ({score}/10) | {t}')

NEGATIVE (5/10) | Není to špatné, splnilo účel.
NEUTRAL (4/10) | Solidní kvalita za rozumnou cenu.
POSITIVE (3/10) | Režisér odvádí skvělou práci, film stojí za to.
POSITIVE (2/10) | Fakt super dobrý film.
NEUTRAL (5/10) | Celkem dobrý, nemám výhrady.
NEGATIVE (3/10) | Nejhorší film co jsem kdy viděl.
POSITIVE (8/10) | Krásný příběh, dojemný a velmi dobře zahraný.
NEUTRAL (5/10) | Není to úplně špatné, ale čekal jsem víc.
NEGATIVE (7/10) | Dalo se to přežít, ale znovu bych do toho nešel.
NEUTRAL (6/10) | Na první pohled dobré, ale rychle to omrzí.
NEGATIVE (7/10) | Nevím, jestli se mi to líbilo, ale neurazilo to.
NEGATIVE (8/10) | Čekal jsem katastrofu, nakonec to ušlo.
NEGATIVE (4/10) | Má to svoje chyby, ale celkově spokojenost.
NEUTRAL (4/10) | Není to nic extra, ale neurazí.
NEGATIVE (5/10) | Občas to drhne, jinak docela dobré.
NEUTRAL (4/10) | Zpracování nic moc, ale myšlenka fajn.
NEGATIVE (7/10) | Herecké výkony slabší, ale příběh to zachraňuje.
NEGATIVE (5/10) | Nadšení se neko

In [16]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-z\u00e1\u010d\u010f\u00e9\u011b\u00ed\u0148\u00f3\u0159\u0161\u0165\u00fa\u016f\u00fd\u017e\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def predict_sentiment(text):
    cleaned = clean_text(text)
    pred = pipeline.predict([cleaned])[0]
    proba = pipeline.predict_proba([cleaned])[0]
    label = 'POSITIVE' if pred == 1 else 'NEGATIVE'

    # Confidence 0.5-1.0 přemapuj na 1-10
    confidence = max(proba)
    score = int(1 + (confidence - 0.5) * 18)  # 0.5→1, 1.0→10
    score = max(1, min(10, score))

    return label, confidence, score

testy2 = [
    'Není to špatné, splnilo účel.',
    'Solidní kvalita za rozumnou cenu.',
    'Režisér odvádí skvělou práci, film stojí za to.',
    'Fakt super dobrý film.',
    'Celkem dobrý, nemám výhrady.',
    'Nejhorší film co jsem kdy viděl.',
    'Krásný příběh, dojemný a velmi dobře zahraný.',

    'Není to úplně špatné, ale čekal jsem víc.',
    'Dalo se to přežít, ale znovu bych do toho nešel.',
    'Na první pohled dobré, ale rychle to omrzí.',
    'Nevím, jestli se mi to líbilo, ale neurazilo to.',
    'Čekal jsem katastrofu, nakonec to ušlo.',
    'Má to svoje chyby, ale celkově spokojenost.',
    'Není to nic extra, ale neurazí.',
    'Občas to drhne, jinak docela dobré.',
    'Zpracování nic moc, ale myšlenka fajn.',
    'Herecké výkony slabší, ale příběh to zachraňuje.',
    'Nadšení se nekonalo, ale úplný propadák to taky není.',
    'Není to špatné, ale ani dobré.',
    'Místy skvělé, místy dost slabé.',
]

for t in testy2:
    label, conf, score = predict_sentiment(t)
    print(f'{label} ({score}/10) | {t}')

NEGATIVE (4/10) | Není to špatné, splnilo účel.
NEGATIVE (1/10) | Solidní kvalita za rozumnou cenu.
NEGATIVE (1/10) | Režisér odvádí skvělou práci, film stojí za to.
NEGATIVE (1/10) | Fakt super dobrý film.
NEGATIVE (1/10) | Celkem dobrý, nemám výhrady.
NEGATIVE (1/10) | Nejhorší film co jsem kdy viděl.
NEGATIVE (4/10) | Krásný příběh, dojemný a velmi dobře zahraný.
NEGATIVE (2/10) | Není to úplně špatné, ale čekal jsem víc.
NEGATIVE (3/10) | Dalo se to přežít, ale znovu bych do toho nešel.
NEGATIVE (3/10) | Na první pohled dobré, ale rychle to omrzí.
NEGATIVE (2/10) | Nevím, jestli se mi to líbilo, ale neurazilo to.
NEGATIVE (7/10) | Čekal jsem katastrofu, nakonec to ušlo.
NEGATIVE (1/10) | Má to svoje chyby, ale celkově spokojenost.
NEGATIVE (2/10) | Není to nic extra, ale neurazí.
NEGATIVE (5/10) | Občas to drhne, jinak docela dobré.
NEGATIVE (2/10) | Zpracování nic moc, ale myšlenka fajn.
NEGATIVE (4/10) | Herecké výkony slabší, ale příběh to zachraňuje.
NEGATIVE (3/10) | Nadšení s

In [23]:
import pickle

with open('model3classes.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

print('Model ulozen jako model3classes.pkl')
files.download('model3classes.pkl')

Model ulozen jako model3classes.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>